### Notebook for trying different things and learning

In [3]:
test_string = "hello! こんにちは!"

In [4]:
print("START")
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)
bytes_list = list(utf8_encoded)
print("In characters ", len(test_string), " number of characters")
print(bytes_list)
print("In UTF8 ", len(bytes_list), " characters")
bytes_list = list(test_string.encode("utf-16"))
print(bytes_list)
print("In UTF16 ", len(bytes_list), " characters")
bytes_list = list(test_string.encode("utf-32"))
print(bytes_list)
print("In UTF32 ", len(bytes_list), " characters")

START
b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
In characters  13  number of characters
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
In UTF8  23  characters
[255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 33, 0, 32, 0, 83, 48, 147, 48, 107, 48, 97, 48, 111, 48, 33, 0]
In UTF16  28  characters
[255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0, 33, 0, 0, 0, 32, 0, 0, 0, 83, 48, 0, 0, 147, 48, 0, 0, 107, 48, 0, 0, 97, 48, 0, 0, 111, 48, 0, 0, 33, 0, 0, 0]
In UTF32  56  characters


In [5]:
print(ord('h'))

104


In [6]:
import regex as re
pat_oai =  r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

text = "Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad."
compiled = re.compile(pat_oai)
matches = compiled.finditer(text)
for match in matches:
    print(match.group())


Once
 upon
 a
 time
,
 in
 a
 warm
 and
 sunny
 place
,
 there
 was
 a
 big
 pit
.
 A
 little
 boy
 named
 Tom
 liked
 to
 play
 near
 the
 pit
.
 One
 day
,
 Tom
 lost
 his
 red
 ball
.
 He
 was
 very
 sad
.


In [7]:
# BPE Implementation
from typing import BinaryIO
import os
import multiprocessing as mp
import regex as re

def find_chunk_boundaries(
        file: BinaryIO,
        desired_num_chunks: int,
        split_special_token: bytes,
    ) -> list[int]:
        """
        Chunk the file into parts that can be counted independently.
        May return fewer chunks if the boundaries end up overlapping.
        split_special_token is used to create "clean" boundaries that will not have interdependence.
        """
        assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"
        print("split special token for chunking is ", split_special_token)

        # Get total file size in bytes
        file.seek(0, os.SEEK_END)
        file_size = file.tell()
        file.seek(0)

        chunk_size = file_size // desired_num_chunks

        # Initial guesses for chunk boundary locations, uniformly spaced
        # Chunks start on previous index, don't include last index
        chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
        chunk_boundaries[-1] = file_size

        mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

        found_chunk_seps = 0
        for bi in range(1, len(chunk_boundaries) - 1):
            initial_position = chunk_boundaries[bi]
            file.seek(initial_position)  # Start at boundary guess
            while True:
                mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

                # If EOF, this boundary should be at the end of the file
                if mini_chunk == b"":
                    chunk_boundaries[bi] = file_size
                    break

                # Find the special token in the mini chunk
                found_at = mini_chunk.find(split_special_token)
                if found_at != -1:
                    chunk_boundaries[bi] = initial_position + found_at
                    found_chunk_seps += 1
                    break
                initial_position += mini_chunk_size

        # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
        return sorted(set(chunk_boundaries))

def _process_chunk(args: tuple[str, int, int, list[str]]) -> list[str]:
    filepath, start, end, special_tokens_str = args
    with open(filepath, "rb") as f:
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")

    pat_oai = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    compiled = re.compile(pat_oai)

    # Split out special tokens before running regex pre-tokenization
    if special_tokens_str:
        pattern = "|".join(re.escape(tok) for tok in special_tokens_str)
        segments = re.split(pattern, chunk)
    else:
        segments = [chunk]

    chunk_tokens = []
    for segment in segments:
        if segment:
            chunk_tokens.extend(compiled.findall(segment))
    return chunk_tokens

def pre_tokenize(
    filepath: str,
    special_tokens: list[bytes],
    doc_separator: bytes,
    num_processes: int = 4,
) -> list[str]:
    with open(filepath, "rb") as f:
        boundaries = find_chunk_boundaries(f, num_processes, doc_separator)
        print("Chunk boundaries: ", boundaries)

    special_tokens_str = [tok.decode("utf-8", errors="ignore") for tok in special_tokens]
    tasks = [
        (filepath, start, end, special_tokens_str)
        for start, end in zip(boundaries[:-1], boundaries[1:])
    ]

    # Use fork context on macOS to avoid spawn __main__ pickling issues in Jupyter
    try:
        ctx = mp.get_context("fork")
    except ValueError:
        ctx = mp.get_context()

    with ctx.Pool(processes=num_processes) as pool:
        chunk_results = pool.map(_process_chunk, tasks)

    pre_tokens = []
    for res in chunk_results:
        pre_tokens.extend(res)
    return pre_tokens

In [7]:
# Observing byte encodings for a special token we create
special_token = bytes("<|endoftext|>", "utf-8")
special_token_encoding = "<|endoftext|>".encode("utf-8")
print(list(special_token))
print(list(special_token_encoding))

[60, 124, 101, 110, 100, 111, 102, 116, 101, 120, 116, 124, 62]
[60, 124, 101, 110, 100, 111, 102, 116, 101, 120, 116, 124, 62]


In [8]:
pre_tokens = pre_tokenize("/Users/dibakarbarua/Desktop/Code/cs336/cs336_basics/data/TinyStoriesV2-GPT4-valid.txt", [special_token], special_token)
print(pre_tokens[0])
print(pre_tokens[1])
print(pre_tokens[2])
print(pre_tokens[3])
print(pre_tokens[-1])
print(pre_tokens[-2])
print(pre_tokens[-3])
print(pre_tokens[-4])

split special token for chunking is  b'<|endoftext|>'
Chunk boundaries:  [0, 5625758, 11252559, 16877372, 22502601]
u
 don
't
 have
.
 happy
 very
 were


In [8]:
%prun pre_tokenize("/Users/dibakarbarua/Desktop/Code/cs336/cs336_basics/data/TinyStoriesV2-GPT4-valid.txt", [special_token], special_token)

split special token for chunking is  b'<|endoftext|>'
Chunk boundaries:  [0, 5625758, 11252559, 16877372, 22502601]
 

         41359 function calls (41241 primitive calls) in 0.665 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      494    0.230    0.000    0.354    0.001 {method 'poll' of 'select.poll' objects}
       20    0.175    0.009    0.176    0.009 {method 'control' of 'select.kqueue' objects}
        6    0.161    0.027    0.161    0.027 {built-in method _pickle.loads}
      572    0.033    0.000    0.033    0.000 {method 'write' of '_io.BytesIO' objects}
       10    0.014    0.001    0.014    0.001 {method 'extend' of 'list' objects}
     1002    0.009    0.000    0.046    0.000 {built-in method posix.waitpid}
        4    0.004    0.001    0.016    0.004 {built-in method posix.fork}
  572/570    0.003    0.000    0.618    0.001 {built-in method posix.read}
        1    0.003    0.003    0.003    0.003 {built-in method _imp.create_dynamic}
        9    0.002    0.000    0.002    0.000 {method 'read' of '_io.BufferedReader' ob

In [9]:
def merge_smol(pre_list, merge_bp_tuple):
    print("pre_list: ", pre_list)
    merge_bp = merge_bp_tuple[0] + merge_bp_tuple[1]
    
    idx = 0
    while idx < len(pre_list):
        if pre_list[idx] == merge_bp_tuple:
            # Update previous pair's second element
            if idx != 0:
                pre_list[idx - 1] = (pre_list[idx - 1][0], merge_bp)
                
            # Update next pair's first element
            if idx != len(pre_list) - 1:
                pre_list[idx + 1] = (merge_bp, pre_list[idx + 1][1])
                
            # Remove the merged pair at idx
            del pre_list[idx]
            # (Do not increment idx here, as elements shifted left)
        else:
            idx += 1
            
    print("post_list: ", pre_list)
    return pre_list

pre_list = [
    (b'b', b'a'),
    (b'a', b'n'),
    (b'n', b'a'),
    (b'a', b'n'),
    (b'n', b'a'),
]
merge_bp = (b'a', b'n')
pre_list = merge_smol(pre_list, merge_bp)
merge_bp = (b'an', b'a')
pre_list = merge_smol(pre_list, merge_bp)
merge_bp = (b'n', b'a')
pre_list = merge_smol(pre_list, merge_bp)

pre_list:  [(b'b', b'a'), (b'a', b'n'), (b'n', b'a'), (b'a', b'n'), (b'n', b'a')]
post_list:  [(b'b', b'an'), (b'an', b'an'), (b'an', b'a')]
pre_list:  [(b'b', b'an'), (b'an', b'an'), (b'an', b'a')]
post_list:  [(b'b', b'an'), (b'an', b'ana')]
pre_list:  [(b'b', b'an'), (b'an', b'ana')]
post_list:  [(b'b', b'an'), (b'an', b'ana')]


In [28]:
from dataclasses import dataclass

def merge(
    filepath: str,
    special_tokens: list[bytes],
    doc_separator: bytes,
    max_vocab_size: int,
):
    ## 1.0 Setup Pre-Tokenization
    special_token = bytes("<|endoftext|>", "utf-8")
    pre_tokens = pre_tokenize(filepath, special_tokens, doc_separator)

    # NOTE: pre_tokens contains a list of all pre-tokens as re.match objects

    ## 2.0 Initialize Vocabulary:
    
    # 2.1 Represent UTF-8 encodings, Integer IDs 0 -> 255
    # NOTE: No character/token representation here, simply integer ID <-> bytes
    utf8_vocab_dict : dict[int, bytes] = {
        byte_val: bytes([byte_val]) for byte_val in range(0, 256)
    }

    # 2.2 Represent special tokens
    # NOTE: Assuming that doc_separator is present in special_tokens
    byte_val_start = 256 # tokens 0-255 allocated to UTF-8
    byte_val_end = byte_val_start + len(special_tokens)
    special_vocab_dict: dict[int, bytes] = {
        byte_val: special_tokens[byte_val - byte_val_start] for byte_val in range(byte_val_start, byte_val_end)
    }

    # 2.3 Represent BPE merged vocabulary
    # Any new token captured will be at the index below and incremented upon capture
    byte_val_start = byte_val_end
    merged_vocab_dict: dict[int, bytes] = {}

    ## 3.0 Initialize Primary Merge Containers
    
    # 3.1 Container to store all pre-tokens and their dynamic tokenized bytearrays from the current vocabulary
    # Example: `alone ` as a pre-token will be initially stored as [b'a', b'l', ..... b' ']
    # Example: `alone ` after adding `al` to the vocabulary will be stored as [b'al', b'o', .... b' '] 
    pretokens_bytes_ : dict[str, list[bytes]] = {
        token : [bytes([val]) for val in token.encode('utf-8')] for token in pre_tokens
    }

    """
    NOTE:
    Reasoning for byte_pair as a tuple of bytes instead of a single bytes object:
    - During Inference / Tokenization, Merges Are Applied Sequentially
    - When a trained BPE tokenizer encodes a new prompt at test time:
        It splits the input into individual bytes.
        It looks up all pairs of adjacent tokens in the input.
        It finds the pair with the highest priority (lowest merge rank) from the learned merges list and replaces $(t_1, t_2)$ with $t_1 t_2$.
            NOTE: This is because both 'th' and 'the' could be in the vocabulary.
            - So the tokenizer needs to have a priority order in the vocabulary to encode into first!
        It repeats until no further merges can be applied.
        If your training aggregated (ab, c) and (a, bc) together, the test-time tokenizer wouldn't know which path to take. Depending on which earlier merges fired on the input text, the text might contain [ab, c] or [a, bc]. If the learned merge rule doesn't match the exact tokens currently present, the merge will fail to trigger, resulting in sub-optimal tokenization and unknown tokens.
    """

    """
    NOTE:
    Data Structure Encapsulation:
    Let's say I have the pre-token 'banana' in my corpus appearing 10 times
    And the word 'and' occuring 100 times
    After 'an' is merged, I need to adjust byte-pair counts in all tokens where 'an' appears!
    I need an additional dict that will store
        pretokens_:
        {
            'banana' : 
            {
                count: 10,
                byte_pairs: [('b', 'a'), ('a', 'n'), .... ]
            }    
        }
    And an additional dict for the actual byte-pair counts and pre-tokens for merging
        bytepair_counts:
        {
            ('b', 'a'): 
            {
                count: 10,
                tokens: ['banana']
            }
            ('a', 'n'): 
            {
                count: 120,
                tokens: ['banana', 'an']
            }
        }
    """
    
    # 3.2 Container to store all pre-tokens and active byte-pairs
    @dataclass
    class PreTokenInfo:
        count: int
        byte_pairs: list[tuple[bytes, bytes]]
    
    @dataclass
    class BytePairInfo:
        count: int
        tokens: list[str]
    pretokens_ : dict[str, PreTokenInfo] = {}
    byte_pairs_ : dict[bytes, BytePairInfo] = {}

    # 3.3 Merge information for bytes merged in-order
    merges_ : list[tuple(bytes, bytes)] = []

    def add_byte_pair_to_token_(token, bps, new_bp):
        if new_bp in bps:
            bps[new_bp].count += 1
            if token not in bps[new_bp].tokens:
                bps[new_bp].tokens.append(token)
        else:
            bps[new_bp] = BytePairInfo(1, [token])
        return bps
    
    def remove_byte_pair_from_token_(bps, old_bp, old_token, token_count):
        assert old_bp in bps
        bps[old_bp].count -= token_count
        if old_token in bps[old_bp].tokens:
            bps[old_bp].tokens.remove(old_token)
    
    def delete_byte_pair_(bps, old_bp):
        print(old_bp)
        assert old_bp in bps
        bps.pop(old_bp)

    # 4.0 Main Loop through all pre-tokens
    # 4.1 Form INITIAL byte-pairs for all tokens
    for token in pre_tokens:
        old_bytes = pretokens_bytes_[token]
        new_bytes = []
        only_one_byte = True
        adding_new_pretoken = token not in pretokens_
        # 4.11 Update counts for pre-token
        if adding_new_pretoken:
            pretokens_[token] = PreTokenInfo(1, [])
        else:
            pretokens_[token].count += 1
        for idx in range(len(old_bytes) - 1):
            if idx == 0:
                only_one_byte = False
            new_byte_pair = (old_bytes[idx], old_bytes[idx + 1])
            new_bytes.append(new_byte_pair)
            # NOTE: For every byte-pair we need to update:
            #   1. Counts for byte-pair
            #   2. Pre-tokens that contain it for merge-traversal
            #       2.a Counts for pre-tokens
            #       2.b Byte-pair list for pre-tokens (first-time observed pre-token)
            
            # 4.12 Update counts and pre-tokens for byte-pair
            byte_pairs_ = add_byte_pair_to_token_(token, byte_pairs_, new_byte_pair)
            # 4.13 Update counts and byte-pairs for pre-tokens
            if not adding_new_pretoken:
                continue
            else:
                pretokens_[token].byte_pairs.append(new_byte_pair)

        # 4.14
        if only_one_byte:
            new_bytes = old_bytes
            # only one byte so we dont have to track any byte-pair in this token
    
    print("Pre-Tokens:")
    # for token in pretokens_:
    #     count = pretokens_[token].count
    #     bps = pretokens_[token].byte_pairs
    #     print(f"{token}: count:{count}, byte-pairs:{bps}")
    print()
    print("Byte-Pairs:")
    # for bp in byte_pairs_:
    #     count = byte_pairs_[bp].count
    #     tokens = byte_pairs_[bp].tokens
    #     print(f"{bp}: count:{count}, tokens:{tokens}")
    print()

    # 5.0 Main Merge Loop
    iter_count = 0
    print("Starting additional vocab at byte-idx = ", byte_val_start)
    while(byte_val_start < (max_vocab_size + 1)):
        """
        5.01 Merge subroutine
        """
        def merge_token_list(pre_list, merge_bp_tuple, token, token_count):
            print("Merging byte-pair ", merge_bp_tuple, " in token ", token, " with previous byte- pairs' pre_list: ", pre_list)
            merge_bp = merge_bp_tuple[0] + merge_bp_tuple[1]
            
            idx = 0
            while idx < len(pre_list):
                if pre_list[idx] == merge_bp_tuple:
                    # Update previous pair's second element
                    if idx != 0:
                        print("Removing instance of byte-pair ", pre_list[idx - 1])
                        remove_byte_pair_from_token_(byte_pairs_, pre_list[idx - 1], token, token_count)
                        pre_list[idx - 1] = (pre_list[idx - 1][0], merge_bp)
                        print("Adding instance of byte-pair ", pre_list[idx - 1])
                        add_byte_pair_to_token_(token, byte_pairs_, pre_list[idx - 1])
                        
                    # Update next pair's first element
                    if idx != len(pre_list) - 1:
                        print("Removing instance of byte-pair ", pre_list[idx + 1])
                        remove_byte_pair_from_token_(byte_pairs_, pre_list[idx + 1], token, token_count)
                        pre_list[idx + 1] = (merge_bp, pre_list[idx + 1][1])
                        print("Adding instance of byte-pair ", pre_list[idx - 1])
                        add_byte_pair_to_token_(token, byte_pairs_, pre_list[idx - 1])
                        
                    # Remove the merged pair at idx
                    del pre_list[idx]
                    # (Do not increment idx here, as elements shifted left)
                else:
                    idx += 1
            return pre_list

        iter_count += 1
        print("Iter: ", iter_count," | Current vocab size = ", byte_val_start)
        # 5.1 Sort byte-pairs by frequency
        # 5.2 Pick merged byte-pair for this iteration
        bytes_, info_ = max(
            byte_pairs_.items(),
            key=lambda item: (item[1].count, item[0])  # higher count wins, then higher byte value
        )
        # 5.3 Create merged 'bytes' object and add to vocab, update vocab pointers
        print()
        print("Merging most frequent byte-pair: ", bytes_)
        merged_vocab_dict[byte_val_start] = bytes_
        byte_val_start += 1
        byte_val_end = byte_val_start
        tokens_in_use = byte_pairs_[bytes_].tokens
        pt_idx = 0
        for pt in tokens_in_use:
            pt_count = pretokens_[pt].count
            bp_list = pretokens_[pt].byte_pairs
            print("Merging pre-token ", pt_idx , "'''", pt, "''' with byte-pairs ", bp_list)
            bp_list = merge_token_list(bp_list, bytes_, pt, pt_count)
            print()
            pt_idx += 1
        print("Deleting merged byte-pair ", bytes_)
        delete_byte_pair_(byte_pairs_, bytes_)
        # 5.4 Update 'byte-pairs' dict with merged bp
            # 5.41 Get token list for merged byte-pair
            # 5.43 Delete byte-pair from dict
        # 5.5 Update 'tokens' dict with merged bp
            # 5.51 Traverse old-byte pair token list
            # 5.52 Get byte-pair list for token for updating
            # 5.53 While traversing the list in case of a match with old byte-pair, 
            #   update l[idx], l[idx-1], l[idx+1]
            #   update counts for above byte-pairs in 'byte-pairs' dict
        print()



special_token = "<|endoftext|>".encode("utf-8")
merge("/Users/dibakarbarua/Desktop/Code/cs336/cs336_basics/data/test_data.txt", [special_token], special_token, 257)

# TODO
# 1. Debug loop state
# 2. Add dict prints for verification

split special token for chunking is  b'<|endoftext|>'
Chunk boundaries:  [0, 34783, 69708, 103964, 138420]
Pre-Tokens:

Byte-Pairs:

Starting additional vocab at byte-idx =  257
Iter:  1  | Current vocab size =  257

Merging most frequent byte-pair:  (b' ', b't')
Merging pre-token  0 '''  to ''' with byte-pairs  [(b' ', b't'), (b't', b'o')]
Merging byte-pair  (b' ', b't')  in token   to  with previous byte- pairs' pre_list:  [(b' ', b't'), (b't', b'o')]
Removing instance of byte-pair  (b't', b'o')
Adding instance of byte-pair  (b' t', b'o')

Merging pre-token  1 '''  the ''' with byte-pairs  [(b' ', b't'), (b't', b'h'), (b'h', b'e')]
Merging byte-pair  (b' ', b't')  in token   the  with previous byte- pairs' pre_list:  [(b' ', b't'), (b't', b'h'), (b'h', b'e')]
Removing instance of byte-pair  (b't', b'h')
Adding instance of byte-pair  (b'h', b'e')

Merging pre-token  2 '''  trust ''' with byte-pairs  [(b' ', b't'), (b't', b'r'), (b'r', b'u'), (b'u', b's'), (b's', b't')]
Merging byte-pa

In [ ]:
    """
    NOTE:
    Data Structure Encapsulation:
    Let's say I have the pre-token 'banana' in my corpus appearing 10 times
    And the word 'and' occuring 100 times
    After 'an' is merged, I need to adjust byte-pair counts in all tokens where 'an' appears!
    I need an additional dict that will store
        pretokens_:
        {
            'banana' : 
            {
                count: 10,
                byte_pairs: [('b', 'a'), ('a', 'n'), ('n', 'a'), ('a', 'n'), ('n', 'a')]
                #### merge 'a', 'n'
                byte_pairs: [('b', 'a'), ('an'), ('n', 'a'), ('an'), ('n', 'a')]
                byte_pairs: [('b', 'an'), ('an', 'a'), ('an'), ('n', 'a')]
                byte_pairs: [('b', 'an'), ('an', 'an'), ('an', 'a')]
                byte_pairs: [('b', 'an'), ('an', 'an'), ('an', 'a')]
                #### merge 'an', 'a'
                byte_pairs: [('b', 'ana'), ('ana', 'n'), ('n', 'a')]
                #### merge 'n', 'a'
                byte_pairs: [('b', 'ana'), ('ana', 'na')]
            }    
        }
    And an additional dict for the actual byte-pair counts and pre-tokens for merging
        bytepair_counts:
        {
            ('b', 'a'): 
            {
                count: 10,
                tokens: ['banana']
            }
            ('a', 'n'): 
            {
                count: 120,
                tokens: ['banana', 'an']
            }
        }
    """